# SCD Type 1: Update without History
In SCD Type 1 we simply update/insert/delete without worring about what was the record earlier it will be simply merge

In [0]:
%sql
CREATE OR REPLACE TABLE sandbox.bronze.customer (
	id BIGINT GENERATED ALWAYS AS IDENTITY,
	c_custkey BIGINT,
	c_name STRING,
	c_address STRING,
	c_phone STRING,
	c_nationkey BIGINT,
	c_mktsegment STRING,
	last_updated_at TIMESTAMP
) USING DELTA;

In [0]:
from delta import DeltaTable
from pyspark.sql.functions import col

In [0]:
ing_df = DeltaTable.forName(spark,"sandbox.ingress.customer").toDF()
ing_df.display()

In [0]:
brnz = DeltaTable.forName(spark, "sandbox.bronze.customer").alias("brz")

brnz.merge(
    ing_df.alias("ing"),
    col("brz.c_custkey")==col("ing.c_custkey"),
).whenMatchedUpdate(
    # Insert Update Exisiting Records
    condition=(
        (col("brz.c_name") != col("ing.c_name")) |
        (col("brz.c_address") != col("ing.c_address")) |
        (col("brz.c_phone") != col("ing.c_phone")) |
        (col("brz.c_nationkey") != col("ing.c_nationkey")) |
        (col("brz.c_mktsegment") != col("ing.c_mktsegment"))
    ),
    set = {
        "c_name": "ing.c_name",
        "c_address": "ing.c_address",
        "c_phone": "ing.c_phone",
        "c_nationkey": "ing.c_nationkey",
        "c_mktsegment": "ing.c_mktsegment",
    }
) \
.whenNotMatchedInsertAll( 
    # Insert New Records
) \
.whenNotMatchedBySourceDelete(
    # Delete Not Matching Records
) \
.execute()
